# llm-finetune-serve — Colab driver

This notebook stays thin on purpose: clone, install, call scripts. All logic
lives in `src/`. If you find yourself writing real code here, it belongs in the
repo instead.

## Two ways to run it

**In the browser:** open this notebook from GitHub, then Runtime → Change
runtime type → GPU.

**From VS Code** (no browser tab): install the official **Google Colab**
extension (publisher: Google), open this file locally, then kernel picker →
`Colab` → `Auto Connect`, and pick a GPU runtime.

Either way the kernel runs on a Colab VM, and the extension does **not** sync
local files to it — so your `src/` edits reach the GPU through GitHub. The loop
is: edit locally → commit + push → re-run the `git pull` cell below → re-run
the stage cell.

In [1]:
!nvidia-smi

Fri Sep  4 22:28:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Clone the repo

Public repo, so no credentials are needed. Re-running this cell pulls the
latest commit rather than re-cloning.

(If you ever flip it back to private, add a GitHub PAT with `repo` scope as a
Colab secret named `GITHUB_TOKEN` — the cell picks it up automatically.)

In [47]:
OWNER = "rushilpatra"
REPO = "llm-finetune-serve"
BRANCH = "main"
REPO_DIR = f"/content/{REPO}"

import os, subprocess

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = os.environ.get("GITHUB_TOKEN")

auth = f"{TOKEN}@" if TOKEN else ""
url = f"https://{auth}github.com/{OWNER}/{REPO}.git"

# Absolute path: a relative one would clone a second copy inside the first
# whenever this cell is re-run after the %cd below.
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
%cd /content/llm-finetune-serve
!git pull --ff-only

/content/llm-finetune-serve
From https://github.com/rushilpatra/llm-finetune-serve
   ca29027..372bce8  main       -> origin/main
Already up to date.


## 2. Install

Colab preinstalls `torch` / `torchvision` / `torchaudio` built against one CUDA
version, and vLLM pulls a torch built against another. Mixing them raises

    RuntimeError: Detected that PyTorch and TorchAudio were compiled with
    different CUDA versions

on `import vllm`. So we uninstall all four — vLLM included, otherwise pip sees
it already installed, skips it, and never reinstalls the torch we just removed
— and let a clean vLLM install pull a matched set.

**This replaces torch, so the kernel must be restarted afterwards.** In the
browser Colab prompts you; **in VS Code it does not** — click the **↺ Restart**
button in the notebook toolbar yourself. Then re-run the clone cell above and
skip straight to the version check; the install is cached.

In [12]:
# vLLM owns the torch stack. vLLM is uninstalled too, so pip actually
# re-resolves it instead of treating the requirement as already satisfied.
!pip uninstall -q -y vllm torch torchvision torchaudio
!pip install -q vllm
!pip install -q -r requirements.txt

# Colab ships torchao 0.10, but PEFT under transformers 5.x refuses anything
# below 0.16 and raises on import. We never quantize, so drop it rather than
# upgrade it and risk disturbing the torch build vLLM just installed.
!pip uninstall -q -y torchao

In [2]:
import torch, transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    # T4 (Turing) has no bf16 support; scripts detect this at runtime.
    print("gpu         ", torch.cuda.get_device_name(0))
    print("bf16        ", torch.cuda.is_bf16_supported())

# Import vLLM here rather than discovering a broken install 20 minutes into a run.
import vllm
print("vllm        ", vllm.__version__)

torch        2.13.0+cu130
transformers 5.16.1
cuda         True
gpu          NVIDIA A100-SXM4-40GB
bf16         True
vllm         0.28.0


## 3. Data smoke test

Prints split sizes, the 8-shot prefix length, and a sample prompt with the
round-trip answer-extraction check.

In [3]:
!python -m src.data --split val --limit 2

split sizes:
  fewshot  8
  val      750
  train    6715
  test     1319

8-shot prefix: 3600 chars

re male. If there are 18 contestants in total, how many of them are male?
Answer: There are 18/3 = 6 female contestants.
There are 18-6 = 12 male contestants.
#### 12

Question: Nancy bought a pie sliced it into 8 pieces. She gave 1/2 to Joe and Darcy, and she gave 1/4 to Carl. How many slices were left?
Answer: The total number of slices she gave to Joe and Darcy is 1/2 x 8 = 4.
The total slice she gave to Carl is 1/4 x 8 = 2.
Therefore, the total slices left is 8 - 4 - 2 = 2.
#### 2

Question: Megan pays $16 for a shirt that costs $22 before sales. What is the amount of the discount?
Answer:

[gold] 6
[extracted from gold completion] 6
[well formed] True
### 12

Question: Nancy bought a pie sliced it into 8 pieces. She gave 1/2 to Joe and Darcy, and she gave 1/4 to Carl. How many slices were left?
Answer: The total number of slices she gave to Joe and Darcy is 1/2 x 8 = 4.
The total s

## 4. Baseline: 8-shot prompting of the base model

Plumbing check first (no GPU, seconds), then the real run. Predictions stream
to `results/baseline_8shot.jsonl` as they are produced, so if the session dies
you re-run the same command and it picks up where it stopped.

In [10]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml --dry-run --limit 4

!! DRY RUN: generations are gold completions, metrics are meaningless

4 examples, 4 already done, 0 to generate

run               baseline_8shot  (val, n=4)
exact match       1.0000 +/- 0.0000
format adherence  1.0000 +/- 0.0000
wall clock        0.3s
gpu               NVIDIA A100-SXM4-40GB  peak 0.45 GB

predictions  results/baseline_8shot-dryrun.jsonl
metrics      results/baseline_8shot-dryrun.metrics.json


In [27]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml

750 examples, 750 already done, 0 to generate

run               baseline_8shot  (val, n=750)
exact match       0.6240 +/- 0.0177
format adherence  0.9667 +/- 0.0066
wall clock        0.3s
gpu               NVIDIA A100-SXM4-40GB  peak 0.45 GB

predictions  results/baseline_8shot.jsonl
metrics      results/baseline_8shot.metrics.json


## 5. Save results off the VM

Colab VMs are ephemeral, and `files.download()` only works from the Colab
browser UI — from a VS Code kernel it silently does nothing. So push `results/`
back to GitHub instead, which versions the predictions alongside the code.

You need a GitHub token with write access to the repo: **Settings → Developer
settings → Personal access tokens → Fine-grained**, repository access limited to
`llm-finetune-serve`, permission **Contents: Read and write**. Either paste it
when prompted, or store it once as a Colab secret named `GITHUB_TOKEN`.

In [49]:
import subprocess
from getpass import getpass

REPO_DIR = "/content/llm-finetune-serve"

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
if not TOKEN:
    TOKEN = getpass("GitHub PAT (Contents: Read and write): ").strip()


def git(*args):
    result = subprocess.run(
        ["git", *args], cwd=REPO_DIR, capture_output=True, text=True
    )
    # Redact: git echoes the remote URL, token and all, on both success and error.
    print((result.stdout + result.stderr).replace(TOKEN, "***").strip())
    return result.returncode


git("config", "user.name", "rushilpatra")
git("config", "user.email", "patra.rushil@gmail.com")
git("add", "results")
# The selected adapter is committed so the repo is self-contained; see README.
git("add", "-f", "outputs/lora_r32_seed0")
git("commit", "-m", "Add eval results from Colab")
REMOTE = f"https://{TOKEN}@github.com/rushilpatra/llm-finetune-serve.git"

# Rebase first: code is usually pushed from the laptop while a run is in
# flight, so the VM is behind by the time it has results to push.
git("pull", "--rebase", REMOTE, "main")
git("push", REMOTE, "HEAD:main")




[main aec125c] Add eval results from Colab
 2 files changed, 876 insertions(+), 876 deletions(-)
 rewrite results/benchmark_vllm_stats_latencies.jsonl (74%)
Current branch main is up to date.
From https://github.com/rushilpatra/llm-finetune-serve
 * branch            main       -> FETCH_HEAD
To https://github.com/rushilpatra/llm-finetune-serve.git
   372bce8..aec125c  HEAD -> main


0

## 6. LoRA fine-tuning

Three cells: a CPU-only formatting check, a tiny GPU run that proves the TRL /
PEFT API works in this environment before committing to the sweep, then the
sweep itself.

The tiny run matters — `transformers` 5.x and TRL move fast, and an API error
is much cheaper to hit after 30 seconds than after six full runs.

In [19]:
!python -m src.train --config configs/lora_r16_seed0.yaml --dry-run --limit 8

8 training examples

--- prompt (loss masked) ---
Question: Bert made 12 sandwiches for his trip. On the first day, he ate half of the sandwiches he made. The next day he ate 2 sandwiches less. How many sandwiches does Bert have left after these two days?
Answer:

--- completion (loss computed) ---
 On the first day, Bert ate 12 / 2 = 6 sandwiches.
The second day he ate 6 - 2 = 4 sandwiches.
So in total Bert is left with 12 - 6 - 4 = 2 sandwiches.
#### 2


In [22]:
!python -m src.train --config configs/lora_r16_seed0.yaml --limit 64

64 training examples
Loading weights: 100% 310/310 [00:00<00:00, 1075.14it/s]
Adding EOS to train dataset: 100% 64/64 [00:00<00:00, 11009.57 examples/s]
Tokenizing train dataset: 100% 64/64 [00:00<00:00, 1090.59 examples/s]
Building labels for train dataset: 100% 64/64 [00:00<00:00, 4383.05 examples/s]
Truncating train dataset: 100% 64/64 [00:00<00:00, 6866.76 examples/s]
Dropping fully masked examples from train dataset: 100% 64/64 [00:00<00:00, 18688.07 examples/s]
trainable params: 10,092,544 / 606,142,464 (1.67%)
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'train_runtime': '5.378', 'train_samples_per_second': '23.8', 'train_steps_per_second': '1.487', 'train_loss': '0.8071', 'entropy': '0.5664', 'num_tokens': '1.934e+04', 'mean_token_accuracy':

### The sweep

Six runs: ranks 8 / 16 / 32 × seeds 0 / 1. Each is trained and then evaluated
zero-shot on the **validation** split — the test split is not touched until a
config has been selected.

Predictions land in `results/eval_lora_<run>.jsonl`, one line per example, which
is what the paired bootstrap consumes later.

In [28]:
RUNS = [f"lora_r{r}_seed{s}" for r in (8, 16, 32) for s in (0, 1)]

for run in RUNS:
    print(f"\n{'=' * 70}\n{run}\n{'=' * 70}")
    !python -m src.train --config configs/{run}.yaml
    !python -m src.evaluate --config configs/eval_lora.yaml --lora outputs/{run} --name eval_{run}


lora_r8_seed0
6715 training examples
Loading weights: 100% 310/310 [00:00<00:00, 1082.86it/s]
Adding EOS to train dataset: 100% 6715/6715 [00:00<00:00, 30846.28 examples/s]
Tokenizing train dataset: 100% 6715/6715 [00:05<00:00, 1256.70 examples/s]
Building labels for train dataset: 100% 6715/6715 [00:00<00:00, 7232.89 examples/s]
Truncating train dataset: 100% 6715/6715 [00:00<00:00, 9743.76 examples/s] 
Dropping fully masked examples from train dataset: 100% 6715/6715 [00:00<00:00, 48861.09 examples/s]
trainable params: 5,046,272 / 601,096,192 (0.84%)
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '0.7236', 'grad_norm': '0.6407', 'learning_rate': '0.00019', 'entropy': '0.6233', 'num_tokens': '4.895e+04', 'mean_token_accuracy': '0.8131', 'epo

## 7. The test split — run once

Config selection is finished. On validation, no rank beat 8-shot prompting on
accuracy (all CIs contain zero); rank 32 had the best seed-averaged accuracy, so
rank 32 is the selected config.

Both of its seeds are evaluated here rather than the better-scoring one. The
selection was of a *rank*; reporting whichever seed got luckier on validation
would bias the headline number upward.

**This is the one-shot final evaluation.** Do not re-run it with a different
config, and do not select anything on the basis of what comes out. ~6 minutes
for all three (1319 questions each).

In [31]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml --split test --name baseline_8shot_test
!python -m src.evaluate --config configs/eval_lora.yaml --split test --lora outputs/lora_r32_seed0 --name eval_lora_r32_seed0_test
!python -m src.evaluate --config configs/eval_lora.yaml --split test --lora outputs/lora_r32_seed1 --name eval_lora_r32_seed1_test

!! TEST SPLIT: this is the one-shot final evaluation

1319 examples, 0 already done, 1319 to generate
INFO 09-05 01:36:17 [api_utils.py:272] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-0.6B-Base'}
INFO 09-05 01:36:19 [model.py:672] Resolved architecture: Qwen3ForCausalLM
INFO 09-05 01:36:19 [model.py:1965] Using max model len 2048
INFO 09-05 01:36:19 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-05 01:36:19 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 09-05 01:36:24 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=5677

## 8. Robustness checks

Three gaps in the result as it stands.

**Exemplar sensitivity.** The fine-tuned side has two seeds; the baseline has
one arbitrary set of eight exemplars and no variance estimate at all. If
swapping exemplars moves the baseline by as much as the fine-tuning effect,
the comparison is weaker than it looks.

**The missing cell.** We have {base, fine-tuned} x {8-shot for base, zero-shot
for fine-tuned}. The fine-tuned model *with* demonstrations completes the 2x2
and separates "fine-tuning taught it the task" from "fine-tuning replaced what
the demonstrations were doing".

All on the test split, reusing the adapter already trained. ~8 minutes.

In [35]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml --split test --fewshot-seed 1 --name baseline_8shot_test_ex1
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml --split test --fewshot-seed 2 --name baseline_8shot_test_ex2
!python -m src.evaluate --config configs/eval_lora.yaml --split test --shots 8 --lora outputs/lora_r32_seed0 --name eval_lora_r32_seed0_8shot_test

!! TEST SPLIT: this is the one-shot final evaluation

1319 examples, 0 already done, 1319 to generate
INFO 09-05 02:33:29 [api_utils.py:272] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-0.6B-Base'}
INFO 09-05 02:33:31 [model.py:672] Resolved architecture: Qwen3ForCausalLM
INFO 09-05 02:33:31 [model.py:1965] Using max model len 2048
INFO 09-05 02:33:31 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-05 02:33:31 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 09-05 02:33:35 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=7156

## 9. Merge the adapter

Folds rank 32 / seed 0 into the base weights. Seed 0 rather than seed 1 because
it is the better of the two on test, and test is the honest measurement.

The merge is verified against the adapter-wrapped model before it is trusted —
a dtype or scaling mistake produces a model that loads cleanly and behaves
differently, which would quietly invalidate the benchmark.

In [39]:
!python -m src.merge --adapter outputs/lora_r32_seed0

merging outputs/lora_r32_seed0 into Qwen/Qwen3-0.6B-Base as torch.bfloat16
Loading weights: 100% 310/310 [00:00<00:00, 1630.97it/s]
Writing model shards: 100% 1/1 [00:01<00:00,  1.22s/it]
merged in 5.6s -> merged/lora_r32_seed0
Loading weights: 100% 310/310 [00:00<00:00, 5814.59it/s]
Loading weights: 100% 310/310 [00:00<00:00, 5560.69it/s]
Loading weights: 100% 310/310 [00:00<00:00, 3858.98it/s]
Loading weights: 100% 310/310 [00:00<00:00, 5760.39it/s]
Loading weights: 100% 310/310 [00:00<00:00, 4404.93it/s]

verification
  merged_vs_adapter  max  25.9375  mean 1.19799  argmax agreement 0.7250
  merged_vs_base     max  31.6875  mean 3.50366  argmax agreement 0.5964
  adapter_vs_base    max  29.7500  mean 3.04589  argmax agreement 0.5821
  adapter is active   True
  merge applied       True
  greedy identical    4/4
  merge verified


## 10. Serving benchmark

Five configurations at concurrency 1 / 8 / 32 / 64. Predictions were registered
in `results/benchmark_predictions.md` before the benchmark was written.

`--verify-adapter` checks the merged weights against base+adapter first and
aborts on mismatch, so a silent dtype or scaling error cannot send us timing the
wrong model.

**Expect 45–70 minutes.** The low-concurrency cells dominate: at concurrency 1,
64 requests run one at a time.

If a vLLM arm fails to allocate memory because a previous engine did not fully
release the GPU, run the arms separately with `--configs <name>`.

In [ ]:
!python -m src.benchmark --verify-adapter outputs/lora_r32_seed0 --requests 64

### Realistic-length check (only if the matrix finished near 45 minutes)

The matrix above forces 256 output tokens; real generations here are ~63 at the
median. This re-runs the two headline vLLM arms at 64 tokens to check the
conclusion survives a production-shaped generation length.

**Skip this if the matrix ran long.** It is a sanity check, not a result.

In [ ]:
!python -m src.benchmark --configs vllm_8shot_apc_on vllm_finetuned \
    --requests 64 --max-new-tokens 64 --out results/benchmark_len64.json

### vLLM diagnostics re-run

The first matrix recorded no prefix cache hit rate and no per-request latencies:
vLLM's offline `LLM` class sets `disable_log_stats=True` by default, which
suppresses both. Without the hit rate, the prefix-caching prediction is only
testable indirectly; without per-request times, every request is charged its
round's completion time and the latency distribution is flat by construction.

This re-runs only the three vLLM arms with stats enabled. ~10 minutes. Results
go to a separate file so the original matrix is not overwritten.

In [48]:
!python -m src.benchmark \
    --configs vllm_8shot_apc_off vllm_8shot_apc_on vllm_finetuned \
    --requests 64 --out results/benchmark_vllm_stats.json


vllm_8shot_apc_off  (Qwen/Qwen3-0.6B-Base, 8-shot)
INFO 09-05 03:50:25 [api_utils.py:272] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'model': 'Qwen/Qwen3-0.6B-Base'}
INFO 09-05 03:50:26 [model.py:672] Resolved architecture: Qwen3ForCausalLM
INFO 09-05 03:50:26 [model.py:1965] Using max model len 2048
INFO 09-05 03:50:26 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-05 03:50:26 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 09-05 03:50:30 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=93338) INFO 09-05 03:50:43 [core.py:122] Initiali

## 11. Serve the merged model on the GPU

The deployment container cannot be run anywhere available to this project: the
laptop is Apple Silicon with no NVIDIA GPU, and Colab has the GPU but is not
where Docker work happens. Running the service here uncontainerized closes most
of that gap — it proves the vLLM backend, engine initialisation and the HTTP
path work against the actual merged weights.

What remains unverified afterwards is only the *combination* of the two: the
GPU image running the GPU code. The README states the three claims separately
rather than implying the whole path was tested.

In [ ]:
import json, os, subprocess, time, urllib.request, urllib.error, signal

env = dict(
    os.environ,
    MODEL_PATH="merged/lora_r32_seed0",
    BACKEND="vllm",
    SHOTS="0",
    MAX_TOKENS="256",
    GPU_MEMORY_UTILIZATION="0.85",
)
log = open("/content/serve.log", "w")
proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "src.serve:app", "--host", "127.0.0.1", "--port", "8000"],
    env=env, stdout=log, stderr=subprocess.STDOUT, cwd="/content/llm-finetune-serve",
)

def get(path, payload=None, timeout=300):
    url = f"http://127.0.0.1:8000{path}"
    data = json.dumps(payload).encode() if payload else None
    headers = {"Content-Type": "application/json"} if payload else {}
    with urllib.request.urlopen(urllib.request.Request(url, data=data, headers=headers), timeout=timeout) as r:
        return json.load(r)

# vLLM engine init takes tens of seconds: weights, CUDA graphs, KV cache.
health = None
for _ in range(120):
    if proc.poll() is not None:
        print(open("/content/serve.log").read()[-3000:])
        raise SystemExit("server exited during startup")
    try:
        health = get("/health", timeout=5)
        if health["status"] == "ok":
            break
    except Exception:
        pass
    time.sleep(2)

print("health:", json.dumps(health))

question = ("Natalia sold clips to 48 of her friends in April, and then she sold "
            "half as many clips in May. How many clips did Natalia sell altogether?")
answer = get("/solve", {"question": question})
print("\nquestion:", question)
print("answer  :", answer["answer"], " (gold 72)")
print("formed  :", answer["well_formed"], "| prompt tokens:", answer["prompt_tokens"],
      "| latency", f"{answer['latency_ms']:.0f} ms")
print("\nreasoning:\n" + answer["reasoning"][:400])

with open("results/serve_check.json", "w") as f:
    json.dump({"health": health, "question": question, "response": answer}, f, indent=2)

proc.send_signal(signal.SIGINT)
proc.wait(timeout=60)
print("\nserver stopped; recorded to results/serve_check.json")

## Next stages

Nothing further runs here. Docker and the final README are laptop work.